# Analyzing proteomics data - NPC2 vs. No NPC2 variants carriers

In [ ]:
# See pipeline on "GBA_PD_proteomics_prep.ipynb" for how proteomics data was extracted, 
# and "extracting_GBA_carriers_ok.ipynb" for how variants were extracted.

## 1) Import proteomics subjects and PLINK2

In [ ]:
# Importing proteomics subjects
import pandas as pd

proteo_subjects = pd.read_csv("/mnt/project/UKBB_participant_IDs_w_proteomics.csv")
proteo_subjects

In [ ]:
# Download plink2
!wget https://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_latest.zip
!unzip plink2_linux_x86_64_latest.zip
!chmod +x plink2

!dx upload plink2 --destination ./

## 2) Extract NPC2 carriers from chr14

In [ ]:
!dx download Bulk/DRAGEN\ WGS/DRAGEN\ population\ level\ WGS\ variants,\ PLINK\ format\ [500k\ release]/ukb24308_c14_b0_v1*

In [ ]:
import pandas as pd
import numpy as np
import os

# Should match pvar "ID" column format
!echo -e 'DRAGEN:chr14:74480701:C:T' > NPC2_splice_donor_variants.snplist
!echo -e 'DRAGEN:chr14:74493217:C:A' > NPC2_stop_gain_variants.snplist
!echo -e 'DRAGEN:chr14:74484507:C:T' > NPC2_D91N_variants.snplist

In [ ]:
# Sanity check for psam file
!head ./ukb24308_c14_b0_v1.psam

In [ ]:
# Re-format .psam
from pathlib import Path

in_path = Path("ukb24308_c14_b0_v1.psam")
out_path = Path("ukb24308_c14_b0_v1.psam.fixed")

with in_path.open("r") as fin, out_path.open("w") as fout:
    # Proper header for PLINK2
    fout.write("#FID\tIID\tPAT\tMAT\tSEX\tPHENOTYPE\n")
    
    for line in fin:
        line = line.strip()
        if not line:
            continue  # skip empty lines
        
        parts = line.split()
        
        # Skip any header-ish line, just in case
        if parts[0].startswith("#FID") or parts[0].startswith("ID_1"):
            continue
        
        # We only care about the first 5 fields; ignore anything extra
        if len(parts) < 5:
            continue  # malformed line, skip
        
        fid, iid, pat, mat, sex = parts[:5]
        phen = "-9"  # force numeric missing phenotype
        
        fout.write(f"{fid}\t{iid}\t{pat}\t{mat}\t{sex}\t{phen}\n")

print("Rebuilt clean ukb24308_c14_b0_v1.psam")

In [ ]:
# Sanity check for psam file
!head ukb24308_c14_b0_v1.psam.fixed

In [ ]:
!mv ukb24308_c14_b0_v1.psam ukb24308_c14_b0_v1_OLD.psam
!mv ukb24308_c14_b0_v1.psam.fixed ukb24308_c14_b0_v1.psam

In [ ]:
# Sanity check for pvar file
!head ukb24308_c14_b0_v1.pvar

In [ ]:
# For bed, bim, fam files
# !./plink2 --bfile ukb24308_c14_b0_v1 \
#          --extract NPC2_splice_donor_variants.snplist \
#          --export A \
#          --out NPC2_splice_donor_risk_variants_raw \
#          --no-categorical

# Using pgen, psam, pvar files:
!./plink2 \
  --pfile ukb24308_c14_b0_v1 \
  --extract NPC2_splice_donor_variants.snplist \
  --export A \
  --out NPC2_splice_donor_variants_raw

In [ ]:
!./plink2 \
  --pfile ukb24308_c14_b0_v1 \
  --extract NPC2_stop_gain_variants.snplist \
  --export A \
  --out NPC2_stop_gain_variants_raw

In [ ]:
!./plink2 \
  --pfile ukb24308_c14_b0_v1 \
  --extract NPC2_D91N_variants.snplist \
  --export A \
  --out NPC2_D91N_variants_raw

In [ ]:
# IMPORTANT - Don't forget to save your result into permanent storage!
!dx upload NPC2_splice_donor_variants_raw.* --destination ./
!dx upload NPC2_stop_gain_variants_raw.* --destination ./
!dx upload NPC2_D91N_variants_raw.* --destination ./

In [ ]:
!head NPC2_splice_donor_variants_raw.raw
!head NPC2_stop_gain_variants_raw.raw
!head NPC2_D91N_variants_raw.raw

## 3) Count number of NPC2 variant carriers with proteomics (irrespective of PD status)

In [ ]:
# Defining a function to extract information (by Jeff Kim)
def extract_alternate_carriers_vectorized(raw_file, include_count=False):
    """
    Process PLINK2 raw format (--recode A) to extract participants carrying alternate alleles
    using vectorized operations for improved performance.
    
    Parameters:
    -----------
    raw_file : str
        Path to the PLINK2 raw format file
    include_count : bool, default=False
        If True, output will include a COUNT column with allele counts
        
    Returns:
    --------
    pd.DataFrame
        DataFrame with columns:
        - IID: Sample identifier
        - VARID: Comma-separated list of variants with alternate alleles
        - COUNT: (Only if include_count=True) Comma-separated list of allele counts
                 corresponding to each variant in VARID
                 0 = Homozygous alternate, 1 = Heterozygous
    """
    # Read the raw file
    df = pd.read_csv(raw_file, sep = r'\s+')
    
    # Get the variant IDs (column names after the first 6 columns)
    variant_cols = df.columns[6:]
    
    # Pre-clean variant names (remove _REF part) - only do this operation once
    clean_variants = np.array([var.split('_')[0] for var in variant_cols])
    
    # Extract genotype data as numpy array for faster processing
    genotype_array = df[variant_cols].values
    
    # Get IIDs as numpy array
    iids = df['IID'].values
    
    # Initialize result dictionaries
    result_dict = {}
    count_dict = {}
    
    # Identify alternate allele carriers (value < 2)
    alt_allele_mask = genotype_array < 2
    
    # Process each sample
    for i in range(len(df)):
        # Find variants where this sample has alternate alleles
        alt_indices = np.where(alt_allele_mask[i])[0]
        
        # Only include samples with at least one alternate allele
        if len(alt_indices) > 0:
            # Get the variant IDs
            alternate_variants = clean_variants[alt_indices]
            
            # Join variant IDs
            result_dict[iids[i]] = ','.join(alternate_variants)
            
            if include_count:
                # Get the actual count values (0 or 1) for these variants
                count_values = genotype_array[i, alt_indices]
                # Convert to strings and join
                count_dict[iids[i]] = ','.join(map(str, count_values))
    
    # Convert the dictionaries to a DataFrame
    if include_count:
        result_df = pd.DataFrame({
            'IID': list(result_dict.keys()),
            'VARID': list(result_dict.values()),
            'COUNT': [count_dict[iid] for iid in result_dict.keys()]
        })
    else:
        result_df = pd.DataFrame(list(result_dict.items()), 
                               columns=['IID', 'VARID'])
    
    return result_df

In [ ]:
# Importing NPC2 carrier information
import numpy as np

NPC2_splice_donor_carriers = extract_alternate_carriers_vectorized("./NPC2_splice_donor_variants_raw.raw", include_count=True)
NPC2_stop_gain_carriers = extract_alternate_carriers_vectorized("./NPC2_stop_gain_variants_raw.raw", include_count=True)
NPC2_D91N_carriers = extract_alternate_carriers_vectorized("./NPC2_D91N_variants_raw.raw", include_count=True)

In [ ]:
# Find NPC2 carriers with proteomics
NPC2_splice_donor_proteo = NPC2_splice_donor_carriers[NPC2_splice_donor_carriers['IID'].isin(proteo_subjects['column'])]
NPC2_splice_donor_proteo

In [ ]:
NPC2_stop_gain_proteo = NPC2_stop_gain_carriers[NPC2_stop_gain_carriers['IID'].isin(proteo_subjects['column'])]
NPC2_stop_gain_proteo

In [ ]:
NPC2_D91N_proteo = NPC2_D91N_carriers[NPC2_D91N_carriers['IID'].isin(proteo_subjects['column'])]
NPC2_D91N_proteo

## 4) Count subjects without NPC2 variants, with proteomics

In [ ]:
## Option 1 - takes too long
# !echo -e 'DRAGEN:chr14:74480701:C:T\nDRAGEN:chr14:74493217:C:A\nDRAGEN:chr14:74484507:C:T' > All_NPC2_risk_variants.snplist

# !./plink2 \
#   --pfile ukb24308_c14_b0_v1 \
#   --exclude All_NPC2_risk_variants.snplist \
#   --export A \
#   --out No_NPC2_variants_raw

## Option 2 - Get list of IDs of everyone with WGS, and remove IDs from people with variants
# To get list of everyone with WGS, use cohort browser or the code below:
!tail -n +2 ukb24308_c14_b0_v1.psam | cut -f1 > WGS_all_samples.txt

In [ ]:
WGS_subjects = pd.read_csv("./WGS_all_samples.txt", header=None)
WGS_subjects

In [ ]:
# Concat all NPC2 carriers
import pandas as pd

all_carriers = pd.concat([
    NPC2_splice_donor_carriers["IID"],
    NPC2_stop_gain_carriers["IID"],
    NPC2_D91N_carriers["IID"]
]).unique()

In [ ]:
# Get non-carriers
No_NPC2_carriers = WGS_subjects[~WGS_subjects[0].isin(all_carriers)]
No_NPC2_carriers

In [ ]:
# Check "weird" IDs
!tail -n 20 ukb24308_c14_b0_v1.psam
!tail -n 20 ukb24308_c14_b0_v1_OLD.psam

In [ ]:
# Remove "weird" IDs
No_NPC2_carriers.columns = ["IID"]

No_NPC2_carriers = No_NPC2_carriers[No_NPC2_carriers["IID"] > 0]
No_NPC2_carriers

In [ ]:
# Get non carriers with proteomics
No_NPC2_proteo = No_NPC2_carriers[No_NPC2_carriers['IID'].isin(proteo_subjects['column'])]
No_NPC2_proteo

FWI - N of subjects with proteomics data:

- NPC2 splice donor: 898
- NPC2 stop gain: 22
- NPC2 D91N: 74
- No NPC2: 51626

Those proportions are expected.
- splice donor is most common out of these, I think it is hypomorphic
- stop gain is a known pathogenic variant, thus true loss of function and the rarest
- D91N is a missense VUS, same one found in our NEUROVC case

In [ ]:
# Save all relevant files into permanent storage
NPC2_splice_donor_proteo.to_csv("NPC2_splice_donor_proteo.txt", sep="\t", index=False)
NPC2_stop_gain_proteo.to_csv("NPC2_stop_gain_proteo.txt", sep="\t", index=False)
NPC2_D91N_proteo.to_csv("NPC2_D91N_proteo.txt", sep="\t", index=False)
No_NPC2_proteo.to_csv("No_NPC2_proteo.txt", sep="\t", index=False)

!dx upload WGS_all_samples* --destination ./
!dx upload NPC2_*.txt --destination ./
!dx upload No_NPC2_*.txt --destination ./

## 5) Basic proteomics preprocessing steps
1. (pre-processing) Filter samples based on metadata
2. (exploration) Examine the distribution of expression per protein
3. (exploration) PCA visualization of the data to determine if there are any outliers
4. (pre-processing) (if needed) Normalization

Sample notebook: https://github.com/dnanexus/UKB_RAP/blob/main/proteomics/protein_DE_analysis/1_preprocess_explore_data.ipynb

In [ ]:
# Rescuing patient information
import os
import pandas as pd
import numpy as np

NPC2_splice_donor_proteo = pd.read_csv("./NPC2_splice_donor_proteo.txt", sep="\t")
NPC2_stop_gain_proteo = pd.read_csv("./NPC2_stop_gain_proteo.txt", sep="\t")
NPC2_D91N_proteo = pd.read_csv("./NPC2_D91N_proteo.txt", sep="\t")
No_NPC2_proteo = pd.read_csv("./No_NPC2_proteo.txt", sep="\t")

NPC2_splice_donor_proteo.head()

In [ ]:
# Rescuing protein abundances
import os
# os.getcwd()
os.chdir("/mnt/project/proteomics_results/")

proteomics = pd.read_csv("complete_proteomics_df.txt", sep="\t") # also in DNAnexus:proteomics_results
proteomics.head(5)

In [ ]:
# Look for NPC2 protein
proteomics.columns = proteomics.columns.str.replace("olink_instance_0.", "", case=False)

matches2 = [col for col in proteomics.columns if col.lower().startswith("npc2")]
matches2

In [ ]:
NPC2_splice_donor_abund = proteomics[proteomics['eid'].isin(NPC2_splice_donor_proteo['IID'])]
NPC2_stop_gain_abund = proteomics[proteomics['eid'].isin(NPC2_stop_gain_proteo['IID'])]
NPC2_D91N_abund = proteomics[proteomics['eid'].isin(NPC2_D91N_proteo['IID'])]
No_NPC2_abund = proteomics[proteomics['eid'].isin(No_NPC2_proteo['IID'])]

In [ ]:
len(NPC2_splice_donor_abund)

In [ ]:
NPC2_splice_donor_abund_cts = NPC2_splice_donor_abund[ ["eid"] + matches2 ]
NPC2_splice_donor_abund_cts.head()

In [ ]:
NPC2_stop_gain_abund_cts = NPC2_stop_gain_abund[ ["eid"] + matches2 ]
NPC2_stop_gain_abund_cts.head()

In [ ]:
NPC2_D91N_abund_cts = NPC2_D91N_abund[ ["eid"] + matches2 ]
NPC2_D91N_abund_cts.head()

In [ ]:
No_NPC2_abund_cts = No_NPC2_abund[ ["eid"] + matches2 ]
No_NPC2_abund_cts.head()

In [ ]:
# Check data distribution
NPC2_splice_donor_abund_cts["npc2"].hist()

In [ ]:
NPC2_stop_gain_abund_cts["npc2"].hist()

In [ ]:
NPC2_D91N_abund_cts["npc2"].hist()

In [ ]:
No_NPC2_abund_cts["npc2"].hist()

In [ ]:
## PCA - Only makes sense if measuring 2 or more proteins

# import pandas as pd
# from sklearn.decomposition import PCA
# from sklearn.preprocessing import StandardScaler
# import matplotlib.pyplot as plt
# import seaborn as sns

# # ---- 1. Put your 4 dfs into a dict with group labels ----
# # Replace the keys/values here with your actual dfs if the names differ
# dfs = {
#     "NPC2_splice_donor": NPC2_splice_donor_abund_cts,
#     "NPC2_stop_gain": NPC2_stop_gain_abund_cts,
#     "NPC2_D91N": NPC2_D91N_abund_cts,
#     "No_NPC2": No_NPC2_abund_cts,
# }

# # ---- 2. Make sure we use the same set of protein columns across all dfs ----
# # (excluding 'eid')
# common_cols = None
# for d in dfs.values():
#     cols = set(d.columns) - {"eid"}
#     common_cols = cols if common_cols is None else (common_cols & cols)

# common_cols = sorted(common_cols)  # for consistent order

# # ---- 3. Concatenate all dfs with a 'group' label ----
# combined_list = []
# for group_name, d in dfs.items():
#     tmp = d[["eid"] + common_cols].copy()
#     tmp["group"] = group_name
#     combined_list.append(tmp)

# combined = pd.concat(combined_list, ignore_index=True)

# # Drop rows with any missing protein value
# combined = combined.dropna(subset=common_cols)

# # ---- 4. Scale features and run PCA on protein columns only ----
# numeric_df = combined[common_cols]

# scaler = StandardScaler()
# scaled_data = scaler.fit_transform(numeric_df)

# pca = PCA(n_components=2)
# pca_result = pca.fit_transform(scaled_data)

# # ---- 5. Build PCA dataframe with group labels for plotting ----
# pca_df = pd.DataFrame(pca_result, columns=["PC1", "PC2"])
# pca_df["group"] = combined["group"].values

# # ---- 6. Plot ----
# plt.figure(figsize=(8, 6))
# sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="group", s=40, alpha=0.8)

# plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
# plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
# plt.title("PCA of 4 groups")
# plt.legend(title="Group")
# plt.tight_layout()
# plt.show()

## 6) Basic "differential expression" analysis
T-tests, without adjusting for covariates. Non-imputed data.

In [ ]:
import pandas as pd
from scipy.stats import ttest_ind
import numpy as np

# Example dataframes
# cases_df and controls_df have the same columns: first column is 'subject_id', others are proteins
# Drop subject ID for analysis
# case_proteins = NPC2_splice_donor_abund_cts.iloc[:, 1:]
# case_proteins = NPC2_stop_gain_abund_cts.iloc[:, 1:]
case_proteins = NPC2_D91N_abund_cts.iloc[:, 1:]
control_proteins = No_NPC2_abund_cts.iloc[:, 1:]

# Prepare output dataframe
results = pd.DataFrame(columns=["protein", "mean_case", "mean_control", "log2FC", "p_value"])

# Loop through each protein
for protein in case_proteins.columns:
    case_values = case_proteins[protein].dropna()
    control_values = control_proteins[protein].dropna()
    
    # t-test
    t_stat, p_val = ttest_ind(case_values, control_values, equal_var=False)
    
    # Means
    mean_case = case_values.mean()
    mean_control = control_values.mean()
    
    # log2 fold change
    # Add small pseudocount if any mean is zero to avoid division by zero
    #log2fc = np.log2((mean_case + 1e-9) / (mean_control + 1e-9))
    log2fc = mean_case - mean_control # data seems to be already log2 transformed
    
    # Append to results
    results = pd.concat([results, pd.DataFrame({
        "protein": [protein],
        "mean_case": [mean_case],
        "mean_control": [mean_control],
        "log2FC": [log2fc],
        "p_value": [p_val]
    })], ignore_index=True)

# Optional: sort by p-value
results = results.sort_values("p_value")

In [ ]:
# Show results
results

In [ ]:
# Plot boxplots of selected proteins
selected_proteins = ["npc2"]

# Melt dataframes to long format
# cases_long = NPC2_splice_donor_abund_cts
# cases_long = NPC2_stop_gain_abund_cts
cases_long = NPC2_D91N_abund_cts
controls_long = No_NPC2_abund_cts

cases_long = cases_long.melt(id_vars="eid", value_vars=selected_proteins, 
                            var_name="protein", value_name="abundance")
cases_long["group"] = "case"

controls_long = controls_long.melt(id_vars="eid", value_vars=selected_proteins, 
                                 var_name="protein", value_name="abundance")
controls_long["group"] = "control"

# Combine
plot_df = pd.concat([cases_long, controls_long], ignore_index=True)

# Replace inf/-inf with NaN
plot_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop rows with NaN
plot_df.dropna(inplace=True)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 10))

sns.boxplot(
    x="protein", y="abundance", hue="group",
    data=plot_df,
    palette="Set2",
    hue_order=["control", "case"],
    ax=ax,
)

sns.stripplot(
    x="protein", y="abundance", hue="group",
    data=plot_df,
    dodge=True,
    jitter=True,
    alpha=0.5,
    color="black",           # marker color
    hue_order=["control", "case"],
    ax=ax,
)

# Fix duplicate legend (take first two handles)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels[:2], title="Group")

ax.set_title("Protein abundance: Case vs Control")
ax.set_ylabel("Abundance")
ax.set_xlabel("Protein")

fig.tight_layout()
fig.savefig("/tmp/NPC2_boxplots_NPC2_D91N_vs_No-NPC.pdf",
            format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Upload final files
import os

outdir = "/tmp"
txt_path = os.path.join(outdir, "t-test_NPC2_protein_NPC2_D91N_vs_No-NPC.txt")

results.to_csv(txt_path, sep="\t", index=False)

!dx upload /tmp/t-test_NPC2_protein_NPC2_D91N_vs_No-NPC.txt --destination proteomics_results/
!dx upload /tmp/NPC2_boxplots_NPC2_D91N_vs_No-NPC.pdf --destination proteomics_results/